# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [ ]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import datasets

### Data Preparation

In [ ]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

In [18]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 11812.46it/s]


In [29]:
MAX_LENGTH = 128

def preprocess_function(examples, tokenizer_):
    result = tokenizer_(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [30]:
qqp_preprocessed = qqp.map(lambda x: preprocess_function(x, tokenizer), batched=True)

Map: 100%|██████████| 390965/390965 [00:24<00:00, 15982.70 examples/s]


In [6]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [22]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set,
    batch_size=32,
    shuffle=False,
    collate_fn=transformers.default_data_collator,
    num_workers=4,
    pin_memory=True,
)

**Task 1 (1 point)**

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [23]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [31]:
def get_accuracy(model, data_loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for batch in data_loader:
            batch = {
                k: v.to(device, non_blocking=True)
                for k, v in batch.items()
                if isinstance(v, torch.Tensor)
            }

            labels = batch.pop("labels")
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(**batch)
            predictions = outputs.logits.argmax(dim=-1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total

    return accuracy

In [26]:
accuracy = get_accuracy(model, val_loader)

In [27]:
assert 0.9 < accuracy < 0.91

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

**Task 2 (4 points)**
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

In [ ]:
import time
model_names = ["textattack/distilbert-base-uncased-QQP", "textattack/bert-base-uncased-QQP", "tanganke/gpt2_qqp"]

for model_name in model_names:
    tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
    model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)
    model = model.to(device)

    qqp_preprocessed = qqp.map(lambda x: preprocess_function(x, tokenizer), batched=True)

    val_set = qqp_preprocessed["validation"]
    val_loader = torch.utils.data.DataLoader(
        val_set,
        batch_size=32,
        shuffle=False,
        collate_fn=transformers.default_data_collator,
        num_workers=4,
        pin_memory=True,
    )
    torch.cuda.synchronize()
    start = time.perf_counter()
    accuracy = get_accuracy(model, val_loader)

    torch.cuda.synchronize()
    end = time.perf_counter()
    speed = len(val_set) / (end - start)

    print(model_name, "\nspeed:", speed, "\naccuracy:", accuracy)

| model                                  | accuracy | speed | size (mb) |
|----------------------------------------|----------|-------|-----------|
| textattack/distilbert-base-uncased-QQP | 0.53379  | 854   | 255       |
| textattack/bert-base-uncased-QQP       | 0.90910  | 545   | 417       |
| tanganke/gpt2_qqp                      | 0.89644  | 265   | 474       |

`textattack/bert-base-uncased-QQP` model shown the best result. `tanganke/gpt2_qqp` model has worse size, speed and accuracy, comparing to the previous one. On the other hand, `textattack/distilbert-base-uncased-QQP` shown best speed but pure accuracy.

### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

**Task 3 (1 point)**
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

In [67]:
train_questions = list(set(qqp["train"]["text1"]))

def find_top_duplicates(question: str, top_k: int = 5):
    result = []

    for start in range(0, len(train_questions), 32):
        candidates = train_questions[start:start + 32]
        inputs = tokenizer(
            [question] * len(candidates),
            candidates,
            padding="max_length",
            max_length=MAX_LENGTH,
            truncation=True,
            return_tensors="pt",
        )
        inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}

        with torch.no_grad():
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(**inputs)

        probabilities = F.softmax(outputs.logits, dim=-1)[:, 1]

        for candidate, probability in zip(candidates, probabilities):
            result.append((candidate, probability.item()))

    result.sort(key=lambda x: x[1], reverse=True)

    return result[:top_k]

In [68]:
find_top_duplicates("What is the meaning of life?")

[('What is actual meaning of life? Indeen, it depend on perception of people or other thing?',
  0.99853515625),
 ("What's are the meaning of life?", 0.99853515625),
 ('What is actual meaning of life?', 0.99853515625),
 ('What the meaning of this all life?', 0.998046875),
 ('What according to you, is the meaning of life?', 0.998046875)]

### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

**Bonus Task 1 (0.5 point)**
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

In [ ]:
<A whole lot of YOUR CODE HERE>

### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

**Bonus Task 2 (1.5 points)**
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part

In [ ]:
<A whole lot of YOUR CODE HERE>